# LeetCode 1844 — Replace All Digits with Characters bằng Python

Link đề: https://leetcode.com/problems/replace-all-digits-with-characters/

Độ khó: **Easy** · Chủ đề: String

Skeleton chữ ký LeetCode:

```python
class Solution:
    def replaceDigits(self, s: str) -> str:

```

**Mục tiêu học**

1. Đọc đúng cấu trúc input: chữ cái ở **chỉ số chẵn**, chữ số ở **chỉ số lẻ**.
2. Hiểu chuỗi Python là **bất biến** (immutable) — không gán `s[i] = ...` được — và các cách vòng qua: `list(s)`, `''.join(...)`.
3. Tự cài đặt hàm `shift(c, x)` bằng `ord()` / `chr()` / `int()`.
4. So sánh 4 cách: nối chuỗi `+=`, sửa trên list, comprehension + `join`, tra bảng chữ cái.
5. Trace, chứng minh, Big-O, lỗi hay gặp và 6 bài tập về "biến đổi ký tự theo vị trí".

## 1. Bài toán yêu cầu gì?

Chuỗi `s` (chỉ số từ 0) có **chữ thường ở chỉ số chẵn** và **chữ số ở chỉ số lẻ**.

Định nghĩa `shift(c, x)` = ký tự đứng **sau** `c` đúng `x` bước trong bảng chữ cái. Ví dụ `shift('a', 5) = 'f'`, `shift('x', 0) = 'x'`.

Với mỗi chỉ số lẻ `i`, thay chữ số `s[i]` bằng `shift(s[i-1], s[i])`. Trả về chuỗi sau khi thay hết. Đề **đảm bảo** kết quả không vượt quá `'z'`.

| s | Kết quả |
|---|---|
| `"a1c1e1"` | `"abcdef"` |
| `"a1b2c3d4e"` | `"abbdcfdhe"` |

**Constraints:** `1 <= s.length <= 100`; chỉ gồm chữ thường và chữ số; `shift(s[i-1], s[i]) <= 'z'`.

Lưu ý: `shift` **không có sẵn** — ta phải tự viết.

In [1]:
examples = [("a1c1e1", "abcdef"), ("a1b2c3d4e", "abbdcfdhe")]
for s, expected in examples:
    for i, ch in enumerate(s):
        kind = "chữ (chẵn)" if i % 2 == 0 else "số (lẻ)"
        print(f"{i}:{ch}:{kind}", end="  ")
    print("\n  ->", expected)

0:a:chữ (chẵn)  1:1:số (lẻ)  2:c:chữ (chẵn)  3:1:số (lẻ)  4:e:chữ (chẵn)  5:1:số (lẻ)  
  -> abcdef
0:a:chữ (chẵn)  1:1:số (lẻ)  2:b:chữ (chẵn)  3:2:số (lẻ)  4:c:chữ (chẵn)  5:3:số (lẻ)  6:d:chữ (chẵn)  7:4:số (lẻ)  8:e:chữ (chẵn)  
  -> abbdcfdhe


## 2. Nền tảng Python

### 2.1. Chỉ số chẵn / lẻ và `range(start, stop, step)`

`i % 2 == 0` kiểm tra chẵn. `range(1, len(s), 2)` sinh `1, 3, 5, ...` — chính là các chỉ số lẻ, nơi có chữ số.

In [2]:
s = "a1b2c3d4e"
print("chỉ số lẻ:", list(range(1, len(s), 2)))
print("chỉ số chẵn:", list(range(0, len(s), 2)))
print("s[::2] =", s[::2], "| s[1::2] =", s[1::2])

chỉ số lẻ: [1, 3, 5, 7]
chỉ số chẵn: [0, 2, 4, 6, 8]
s[::2] = abcde | s[1::2] = 1234


Slicing `s[1::2]` = "bắt đầu từ 1, đi đến hết, bước 2" — lấy mọi ký tự ở chỉ số lẻ.

### 2.2. `ord()`, `chr()` — dịch ký tự bằng số

Máy tính lưu ký tự bằng mã số Unicode: `ord('a') = 97`, `ord('b') = 98`, … Dịch `c` đi `x` bước = `chr(ord(c) + x)`.

In [3]:
print(ord("a"), ord("b"), ord("z"))
print("shift('a', 5) =", chr(ord("a") + 5))
print("shift('x', 0) =", chr(ord("x") + 0))

97 98 122
shift('a', 5) = f
shift('x', 0) = x


### 2.3. `int('3')` — chữ số dạng chuỗi thành số

`s[i]` là **chuỗi** `'3'`, không phải số `3`. Cộng thẳng `ord(c) + '3'` sẽ lỗi `TypeError`. Phải đổi bằng `int(s[i])`.

In [4]:
d = "3"
print(type(d), "->", type(int(d)), int(d) + 1)
try:
    ord("a") + d
except TypeError as e:
    print("Lỗi:", e)

<class 'str'> -> <class 'int'> 4
Lỗi: unsupported operand type(s) for +: 'int' and 'str'


`try / except` bắt lỗi để notebook chạy tiếp — chỉ dùng ở đây để **quan sát** lỗi.

Một cách khác: `ord('3') - ord('0') = 3` (mã các chữ số `'0'..'9'` liên tiếp nhau).

In [5]:
for d in "0379":
    print(d, "->", ord(d) - ord("0"), "| int:", int(d))

0 -> 0 | int: 0
3 -> 3 | int: 3
7 -> 7 | int: 7
9 -> 9 | int: 9


### 2.4. Chuỗi bất biến (immutable)

Trong Python **không** sửa trực tiếp `s[1] = 'b'` được. Có hai cách vượt qua:

1. Đổi sang list: `chars = list(s)`, sửa `chars[i]`, rồi ghép lại `''.join(chars)`.
2. Xây dựng **chuỗi mới** từng phần.

In [6]:
s = "a1c1"
try:
    s[1] = "b"
except TypeError as e:
    print("Lỗi:", e)

chars = list(s)
print("list(s) =", chars)
chars[1] = "b"
print("sau khi sửa:", chars, "->", "".join(chars))

Lỗi: 'str' object does not support item assignment
list(s) = ['a', '1', 'c', '1']
sau khi sửa: ['a', 'b', 'c', '1'] -> abc1


### 2.5. `''.join(list)`

`sep.join(items)` nối các chuỗi trong `items`, chen `sep` ở giữa. Với `sep = ''` thì nối liền.

In [7]:
print("".join(["a", "b", "c"]))
print("-".join(["a", "b", "c"]))

abc
a-b-c


### 2.6. Viết hàm `shift` trước

Tách việc nhỏ thành hàm riêng giúp code chính dễ đọc và dễ test.

In [8]:
def shift(c: str, x: int) -> str:
    return chr(ord(c) + x)

assert shift("a", 5) == "f"
assert shift("x", 0) == "x"
assert shift("y", 1) == "z"
print("shift OK:", shift("a", 1), shift("c", 3), shift("d", 4))

shift OK: b f h


## 3. Cách 1 — nối chuỗi bằng `+=` (trực giác)

Duyệt từng chỉ số. Chẵn ⇒ giữ nguyên ký tự; lẻ ⇒ ghép `shift(s[i-1], int(s[i]))`.

In [9]:
def replace_digits_concat(s: str) -> str:
    res = ""
    for i in range(len(s)):
        if i % 2 == 0:
            res += s[i]
        else:
            res += shift(s[i - 1], int(s[i]))
    return res

for s, expected in examples:
    print(s, "->", replace_digits_concat(s), "| expected", expected)
    assert replace_digits_concat(s) == expected

a1c1e1 -> abcdef | expected abcdef
a1b2c3d4e -> abbdcfdhe | expected abbdcfdhe


### 3.1. Nút thắt (về lý thuyết) của `+=` trên chuỗi

Vì chuỗi bất biến, `res += ch` về nguyên tắc tạo **chuỗi mới** dài thêm 1 — có thể sao chép lại toàn bộ `res` mỗi lần, dẫn tới `O(n²)` trong trường hợp xấu. CPython thường tối ưu được trường hợp này, nhưng **không được đảm bảo** (ở ngôn ngữ/trình thông dịch khác thì không).

Với `n <= 100` không ảnh hưởng gì, nhưng thói quen tốt là: **gom vào list rồi `join` một lần** — luôn `O(n)`.

In [10]:
import timeit
def concat_build(n):
    r = ""
    for _ in range(n):
        r += "a"
    return r

def join_build(n):
    return "".join(["a" for _ in range(n)])

n = 20000
print("+=   :", round(timeit.timeit(lambda: concat_build(n), number=20), 4), "giây (tùy máy)")
print("join :", round(timeit.timeit(lambda: join_build(n), number=20), 4), "giây (tùy máy)")

+=   : 0.0297 giây (tùy máy)
join : 0.0157 giây (tùy máy)


Con số thời gian phụ thuộc máy; điều cần nhớ là `join` có độ phức tạp được **đảm bảo** tuyến tính.

### 3.2. Insight

- Chỉ ký tự ở chỉ số **lẻ** thay đổi; chữ ở chỉ số chẵn **giữ nguyên**.
- Ký tự mới tại `i` chỉ phụ thuộc `s[i-1]` (gốc — luôn là chữ, không bị thay) và `s[i]`.
- Vì vậy mỗi vị trí lẻ xử lý **độc lập**, không có thứ tự phụ thuộc — một lượt duyệt là đủ, có thể sửa tại chỗ trên list mà không sợ "đọc nhầm giá trị đã sửa".

In [11]:
s = "a1b2c3d4e"
for i in range(1, len(s), 2):
    print(f"i={i}: s[i-1]={s[i-1]!r} là chữ? {s[i-1].isalpha()}, s[i]={s[i]!r} là số? {s[i].isdigit()}")

i=1: s[i-1]='a' là chữ? True, s[i]='1' là số? True
i=3: s[i-1]='b' là chữ? True, s[i]='2' là số? True
i=5: s[i-1]='c' là chữ? True, s[i]='3' là số? True
i=7: s[i-1]='d' là chữ? True, s[i]='4' là số? True


`str.isalpha()` / `str.isdigit()` kiểm tra ký tự là chữ cái / chữ số.

## 4. Cách 2 — lời giải chính: sửa trên list rồi `join`

Đoạn 1: `chars = list(s)`.

Đoạn 2: `for i in range(1, len(s), 2)` — chỉ đi qua chỉ số lẻ.

Đoạn 3: `chars[i] = shift(chars[i-1], int(chars[i]))` — `chars[i-1]` là chữ ở chỉ số chẵn, **chưa bao giờ bị sửa**.

Đoạn 4: `return ''.join(chars)`.

In [12]:
def replace_digits_list(s: str) -> str:
    chars = list(s)
    for i in range(1, len(chars), 2):
        chars[i] = shift(chars[i - 1], int(chars[i]))
    return "".join(chars)

for s, expected in examples:
    assert replace_digits_list(s) == expected
print(replace_digits_list("a1c1e1"), replace_digits_list("a1b2c3d4e"))

abcdef abbdcfdhe


Chuỗi độ dài lẻ (kết thúc bằng chữ, như `"a1b2c3d4e"`) hay chẵn (kết thúc bằng số, như `"a1c1e1"`) đều đúng: `range(1, len, 2)` tự dừng đúng chỗ.

In [13]:
for s in ["a", "a0", "a1b", "z0"]:
    print(repr(s), "-> chỉ số lẻ:", list(range(1, len(s), 2)), "->", replace_digits_list(s))

'a' -> chỉ số lẻ: [] -> a
'a0' -> chỉ số lẻ: [1] -> aa
'a1b' -> chỉ số lẻ: [1] -> abb
'z0' -> chỉ số lẻ: [1] -> zz


## 5. TRACE từng bước

Trace `s = "a1b2c3d4e"`:

| i | s[i-1] | s[i] | ord(s[i-1]) | + int | chr | chars sau bước |
|---:|:---:|:---:|---:|---:|:---:|---|
| 1 | a | 1 | 97 | 98 | b | a b b 2 c 3 d 4 e |
| 3 | b | 2 | 98 | 100 | d | a b b d c 3 d 4 e |
| 5 | c | 3 | 99 | 102 | f | a b b d c f d 4 e |
| 7 | d | 4 | 100 | 104 | h | a b b d c f d h e |

Kết quả: `"abbdcfdhe"`.

In [14]:
def trace_replace_digits(s: str) -> str:
    chars = list(s)
    print("ban đầu:", " ".join(chars))
    for i in range(1, len(chars), 2):
        prev, digit = chars[i - 1], int(chars[i])
        new = chr(ord(prev) + digit)
        print(f"i={i}: shift({prev!r}, {digit}) = chr({ord(prev)} + {digit}) = chr({ord(prev) + digit}) = {new!r}")
        chars[i] = new
        print("        ->", " ".join(chars))
    res = "".join(chars)
    print("kết quả:", res)
    return res

assert trace_replace_digits("a1b2c3d4e") == "abbdcfdhe"

ban đầu: a 1 b 2 c 3 d 4 e
i=1: shift('a', 1) = chr(97 + 1) = chr(98) = 'b'
        -> a b b 2 c 3 d 4 e
i=3: shift('b', 2) = chr(98 + 2) = chr(100) = 'd'
        -> a b b d c 3 d 4 e
i=5: shift('c', 3) = chr(99 + 3) = chr(102) = 'f'
        -> a b b d c f d 4 e
i=7: shift('d', 4) = chr(100 + 4) = chr(104) = 'h'
        -> a b b d c f d h e
kết quả: abbdcfdhe


Trace thứ hai: chữ số `0` (ký tự lặp lại chính nó) và dịch tới sát `'z'`.

In [15]:
assert trace_replace_digits("x0y1q9") == "xxyzqz"

ban đầu: x 0 y 1 q 9
i=1: shift('x', 0) = chr(120 + 0) = chr(120) = 'x'
        -> x x y 1 q 9
i=3: shift('y', 1) = chr(121 + 1) = chr(122) = 'z'
        -> x x y z q 9
i=5: shift('q', 9) = chr(113 + 9) = chr(122) = 'z'
        -> x x y z q z
kết quả: xxyzqz


## 6. Cách 3 — comprehension + `join` (một biểu thức)

Dùng **biểu thức điều kiện** `A if cond else B` bên trong comprehension: chỉ số chẵn lấy `s[i]`, chỉ số lẻ lấy ký tự dịch.

In [16]:
def replace_digits_comprehension(s: str) -> str:
    return "".join(
        s[i] if i % 2 == 0 else chr(ord(s[i - 1]) + int(s[i]))
        for i in range(len(s))
    )

print(replace_digits_comprehension("a1b2c3d4e"))

abbdcfdhe


### 6.1. Biến thể: `zip` cặp (chữ, số)

`zip(s[::2], s[1::2])` ghép từng chữ ở chỉ số chẵn với chữ số **ngay sau nó**. Nếu chuỗi dài lẻ, chữ cuối không có cặp nên phải thêm riêng.

In [17]:
s = "a1b2c3d4e"
print(list(zip(s[::2], s[1::2])))

def replace_digits_zip(s: str) -> str:
    parts = []
    for c, d in zip(s[::2], s[1::2]):
        parts.append(c + chr(ord(c) + int(d)))
    if len(s) % 2 == 1:
        parts.append(s[-1])        # chữ cuối không có chữ số đi kèm
    return "".join(parts)

for s, expected in examples:
    assert replace_digits_zip(s) == expected
print(replace_digits_zip("a1b2c3d4e"))

[('a', '1'), ('b', '2'), ('c', '3'), ('d', '4')]
abbdcfdhe


`zip` **dừng ở dãy ngắn hơn** — đây là lý do phải xử lý chữ cuối riêng khi độ dài lẻ.

## 7. Cách 4 — tra bảng chữ cái

Thay vì dùng mã Unicode, dùng vị trí trong `string.ascii_lowercase`: `ALPHA.index(c)` cho vị trí 0..25, rồi `ALPHA[pos + x]`. Cách này dễ mở rộng nếu muốn **vòng lại** (`% 26`) ở biến thể.

In [18]:
import string
ALPHA = string.ascii_lowercase

def replace_digits_table(s: str) -> str:
    pos = {ch: i for i, ch in enumerate(ALPHA)}   # dict chữ -> vị trí, tra O(1)
    chars = list(s)
    for i in range(1, len(s), 2):
        chars[i] = ALPHA[pos[s[i - 1]] + int(s[i])]
    return "".join(chars)

print(pos_demo := {ch: i for i, ch in enumerate("abc")})
print(replace_digits_table("a1b2c3d4e"))

{'a': 0, 'b': 1, 'c': 2}
abbdcfdhe


`{ch: i for ...}` là **dict comprehension**. Dấu `:=` ("walrus") gán và trả giá trị trong cùng biểu thức — chỉ dùng cho tiện in ở đây.

## 8. Code nộp LeetCode

### Cách A — list + join (khuyên dùng)

In [19]:
class Solution:
    def replaceDigits(self, s: str) -> str:
        chars = list(s)
        for i in range(1, len(chars), 2):
            chars[i] = chr(ord(chars[i - 1]) + int(chars[i]))
        return "".join(chars)

print(Solution().replaceDigits("a1c1e1"), Solution().replaceDigits("a1b2c3d4e"))

abcdef abbdcfdhe


### Cách B — một biểu thức comprehension

In [20]:
class SolutionComprehension:
    def replaceDigits(self, s: str) -> str:
        return "".join(
            c if i % 2 == 0 else chr(ord(s[i - 1]) + int(c))
            for i, c in enumerate(s)
        )

print(SolutionComprehension().replaceDigits("a1b2c3d4e"))

abbdcfdhe


### Cách C — có hàm `shift` riêng như đề gợi ý

In [21]:
class SolutionShift:
    def shift(self, c: str, x: int) -> str:
        return chr(ord(c) + x)

    def replaceDigits(self, s: str) -> str:
        res = []
        for i, c in enumerate(s):
            res.append(c if i % 2 == 0 else self.shift(s[i - 1], int(c)))
        return "".join(res)

print(SolutionShift().replaceDigits("a1b2c3d4e"))

abbdcfdhe


## 9. Vì sao đúng?

- Ký tự ở chỉ số chẵn không bao giờ bị ghi ⇒ ở output chúng giữ nguyên (đúng đề).
- Với mỗi chỉ số lẻ `i`, ta ghi đúng **một lần** giá trị `chr(ord(s[i-1]) + int(s[i]))`. Vì `s[i-1]` ở chỉ số chẵn và chưa từng bị sửa, giá trị đọc được là **giá trị gốc** ⇒ đúng định nghĩa `shift(s[i-1], s[i])`.
- `chr(ord(c) + x)` đúng là "ký tự thứ `x` sau `c`" vì các chữ `a..z` có mã Unicode **liên tiếp** 97..122; đề đảm bảo không vượt `'z'`.

**Invariant (cách list):** trước vòng lặp với chỉ số `i`, mọi chỉ số lẻ `< i` đã được thay đúng, mọi chỉ số `>= i` còn nguyên như `s`.

## 10. Độ phức tạp

Gọi `n = len(s)`.

| Cách | Time | Space (phụ) |
|---|---|---|
| `+=` nối chuỗi | `O(n)` thường gặp ở CPython, `O(n²)` xấu nhất về lý thuyết | `O(n)` |
| list + join | `O(n)` | `O(n)` |
| comprehension + join | `O(n)` | `O(n)` |
| tra bảng | `O(n)` (+ dựng dict 26 phần tử) | `O(n)` |

Output đã dài `n` nên `O(n)` bộ nhớ là không tránh được.

## 11. Bộ test

Nhóm: mẫu đề; một ký tự; kết thúc bằng chữ số; chữ số 0; chạm `'z'`; chữ số 9; ngẫu nhiên hợp lệ đối chiếu mọi cách.

In [22]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<20} PASS ({actual!r})")

sol = Solution()
check("mẫu 1", sol.replaceDigits("a1c1e1"), "abcdef")
check("mẫu 2", sol.replaceDigits("a1b2c3d4e"), "abbdcfdhe")
check("1 ký tự", sol.replaceDigits("q"), "q")
check("kết thúc bằng số", sol.replaceDigits("m0"), "mm")
check("chạm z", sol.replaceDigits("y1z0"), "yzzz")
check("chữ số 9", sol.replaceDigits("a9b9"), "ajbk")

mẫu 1                PASS ('abcdef')
mẫu 2                PASS ('abbdcfdhe')
1 ký tự              PASS ('q')
kết thúc bằng số     PASS ('mm')
chạm z               PASS ('yzzz')
chữ số 9             PASS ('ajbk')


In [23]:
import random
rng = random.Random(1844)

def random_valid(n):
    out = []
    for i in range(n):
        if i % 2 == 0:
            out.append(rng.choice(ALPHA))
        else:
            c = out[-1]
            out.append(str(rng.randint(0, min(9, ord("z") - ord(c)))))
    return "".join(out)

impls = [replace_digits_concat, replace_digits_list, replace_digits_comprehension, replace_digits_zip,
         replace_digits_table, Solution().replaceDigits, SolutionComprehension().replaceDigits,
         SolutionShift().replaceDigits]
for _ in range(800):
    s = random_valid(rng.randint(1, 100))
    outs = {f(s) for f in impls}
    assert len(outs) == 1, (s, outs)
    res = outs.pop()
    assert res.isalpha() and res[::2] == s[::2]
print("8 cài đặt đồng ý trên 800 chuỗi ngẫu nhiên PASS")

8 cài đặt đồng ý trên 800 chuỗi ngẫu nhiên PASS


Hàm sinh `random_valid` tôn trọng ràng buộc đề (`shift <= 'z'`). Kiểm thêm tính chất: output toàn chữ cái và các chữ ở chỉ số chẵn không đổi.

## 12. Lỗi người mới hay gặp

### Lỗi 1: gán trực tiếp vào chuỗi

In [24]:
def wrong_assign(s):
    for i in range(1, len(s), 2):
        s[i] = chr(ord(s[i - 1]) + int(s[i]))
    return s

try:
    wrong_assign("a1")
except TypeError as e:
    print("Lỗi:", e)

Lỗi: 'str' object does not support item assignment


Sửa: `chars = list(s)` rồi sửa `chars`, cuối cùng `''.join(chars)`.

### Lỗi 2: quên `int()` — cộng mã của chữ số

In [25]:
def wrong_ord_digit(s):
    chars = list(s)
    for i in range(1, len(chars), 2):
        chars[i] = chr(ord(chars[i - 1]) + ord(chars[i]))   # ord('1') = 49!
    return "".join(chars)

print("sai :", repr(wrong_ord_digit("a1")), "| đúng:", repr(Solution().replaceDigits("a1")))

sai : 'a\x92' | đúng: 'ab'


`ord('1') = 49` nên `97 + 49 = 146` — một ký tự lạ. Sửa: `int(chars[i])` hoặc `ord(chars[i]) - ord('0')`.

### Lỗi 3: nhầm chẵn/lẻ — duyệt `range(0, n, 2)`

In [26]:
def wrong_even_indices(s):
    chars = list(s)
    for i in range(0, len(chars), 2):            # chỉ số CHẴN là chữ cái!
        chars[i] = chr(ord(chars[i - 1]) + int(chars[i]))
    return "".join(chars)

try:
    wrong_even_indices("a1c1")
except ValueError as e:
    print("Lỗi:", e)

Lỗi: invalid literal for int() with base 10: 'a'


`int('a')` báo `ValueError`; thêm nữa với `i = 0`, `chars[i - 1]` là `chars[-1]` — ký tự **cuối** chuỗi. Sửa: chữ số nằm ở chỉ số **lẻ** ⇒ `range(1, len(s), 2)`.

### Lỗi 4: dùng `zip` mà quên chữ cuối khi độ dài lẻ

In [27]:
def wrong_zip(s):
    return "".join(c + chr(ord(c) + int(d)) for c, d in zip(s[::2], s[1::2]))

print("sai :", wrong_zip("a1b2c3d4e"), "| đúng:", Solution().replaceDigits("a1b2c3d4e"))

sai : abbdcfdh | đúng: abbdcfdhe


Mất chữ `e` cuối. Sửa: nếu `len(s) % 2 == 1` thì thêm `s[-1]`.

## 13. Mẹo nhận dạng pattern

- "Thay ký tự tại vị trí theo quy tắc" + chuỗi Python ⇒ **`list(s)` → sửa → `''.join`**.
- "Dịch chữ cái `k` bước" ⇒ `chr(ord(c) + k)`; có vòng lại thì `chr((ord(c) - 97 + k) % 26 + 97)` (mật mã Caesar).
- Chữ số dạng ký tự ⇒ luôn `int(ch)` hoặc `ord(ch) - ord('0')`.
- Cấu trúc xen kẽ chẵn/lẻ ⇒ `range(1, n, 2)` hoặc slicing `s[::2]`, `s[1::2]`.
- Giá trị mới chỉ phụ thuộc **dữ liệu gốc không bị sửa** ⇒ sửa tại chỗ an toàn.

## 14. Biến thể

1. **Có vòng lại**: cho phép vượt `'z'` thì quay về `'a'` (Caesar cipher).
2. **Dịch lùi**: `shift(c, -x)`.
3. **Mã hóa ngược**: từ chuỗi kết quả, khôi phục chuỗi có chữ số (với giả định khoảng cách 0..9).
4. **LeetCode 2194 / 848 Shifting Letters**: dịch theo tổng dồn — cần prefix sum.

In [28]:
def shift_wrap(c, x):
    return chr((ord(c) - ord("a") + x) % 26 + ord("a"))

def encode_back(t):
    # t có dạng chữ-chữ-chữ..., khôi phục chữ số ở chỉ số lẻ
    chars = list(t)
    for i in range(1, len(t), 2):
        chars[i] = str(ord(t[i]) - ord(t[i - 1]))
    return "".join(chars)

print("shift_wrap('y', 3) =", shift_wrap("y", 3))
print("encode_back('abbdcfdhe') =", encode_back("abbdcfdhe"))
assert encode_back(Solution().replaceDigits("a1b2c3d4e")) == "a1b2c3d4e"

shift_wrap('y', 3) = b
encode_back('abbdcfdhe') = a1b2c3d4e


## 15. Bài tập (tự làm trước)

**Bài 1 — tính tay:** `replaceDigits("k2m5p0")` = ?

**Bài 2 — dự đoán output:**

```python
s = "abcdef"
print(s[1::2], s[::2], list(range(1, len(s), 2)))
```

**Bài 3 — điền TODO:** lời giải list + join.

In [29]:
def exercise_replace_digits(s):
    # TODO 1: chars = list(s)
    # TODO 2: duyệt i = 1, 3, 5, ...
    # TODO 3: chars[i] = chr(ord(chars[i-1]) + int(chars[i]))
    # TODO 4: trả về ''.join(chars)
    pass

# assert exercise_replace_digits("a1c1e1") == "abcdef"
# assert exercise_replace_digits("a1b2c3d4e") == "abbdcfdhe"

**Bài 4 — biến thể Caesar:** viết `exercise_caesar(text, k)` dịch **mọi** chữ thường đi `k` bước, có vòng lại.

In [30]:
def exercise_caesar(text, k):
    # TODO: dùng % 26
    pass

# assert exercise_caesar("xyz", 3) == "abc"
# assert exercise_caesar("abc", 26) == "abc"

**Bài 5 — khôi phục:** viết `exercise_encode(t)` nhận chuỗi kết quả (độ dài bất kỳ) và trả về chuỗi gốc có chữ số, giả sử mọi khoảng cách nằm trong `0..9`.

In [31]:
def exercise_encode(t):
    # TODO: ở chỉ số lẻ, chữ số = ord(t[i]) - ord(t[i-1])
    pass

# assert exercise_encode("abcdef") == "a1c1e1"
# assert exercise_encode("abbdcfdhe") == "a1b2c3d4e"

**Bài 6 — mini challenge (Shifting Letters):** cho chuỗi `s` và list `shifts` cùng độ dài; với mỗi `i`, dịch **`i+1` ký tự đầu** của `s` đi `shifts[i]` bước (có vòng). Trả về chuỗi cuối. Ví dụ `s="abc", shifts=[3,5,9]` → `"rpl"`.

In [32]:
def exercise_shifting_letters(s, shifts):
    # TODO: ký tự j bị dịch tổng shifts[j] + shifts[j+1] + ... -> duyệt ngược cộng dồn
    pass

# assert exercise_shifting_letters("abc", [3, 5, 9]) == "rpl"
# assert exercise_shifting_letters("aaa", [1, 2, 3]) == "gfd"

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | chỉ đổi chỉ số lẻ | `k+2`, `m+5`, `p+0` | `m`, `r`, `p` |
| 2 | slicing bước 2 | bắt đầu ở 1 hay 0 | `"bdf" "ace" [1, 3, 5]` |
| 3 | list vì chuỗi bất biến | `range(1, len(chars), 2)` | `chr(ord(prev) + int(d))` |
| 4 | đổi về 0..25 | `(ord(c) - 97 + k) % 26` | `+ 97` rồi `chr` |
| 5 | ngược với shift | hiệu hai mã | `str(...)` để thành ký tự |
| 6 | tổng hậu tố | duyệt `i` từ cuối, `total += shifts[i]` | `chr((ord(s[i]) - 97 + total) % 26 + 97)` |

## 17. Đáp án tham khảo

**Bài 1:** `"kmmrpp"`.

**Bài 2:** `bdf ace [1, 3, 5]`.

In [33]:
print(Solution().replaceDigits("k2m5p0"))
s = "abcdef"
print(s[1::2], s[::2], list(range(1, len(s), 2)))

kmmrpp
bdf ace [1, 3, 5]


In [34]:
def answer_replace_digits(s):
    chars = list(s)
    for i in range(1, len(chars), 2):
        chars[i] = chr(ord(chars[i - 1]) + int(chars[i]))
    return "".join(chars)


def answer_caesar(text, k):
    return "".join(chr((ord(c) - 97 + k) % 26 + 97) for c in text)


def answer_encode(t):
    chars = list(t)
    for i in range(1, len(t), 2):
        chars[i] = str(ord(t[i]) - ord(t[i - 1]))
    return "".join(chars)


def answer_shifting_letters(s, shifts):
    res = [""] * len(s)
    total = 0
    for i in range(len(s) - 1, -1, -1):
        total += shifts[i]
        res[i] = chr((ord(s[i]) - 97 + total) % 26 + 97)
    return "".join(res)


assert answer_replace_digits("a1c1e1") == "abcdef"
assert answer_replace_digits("a1b2c3d4e") == "abbdcfdhe"
assert answer_caesar("xyz", 3) == "abc"
assert answer_caesar("abc", 26) == "abc"
assert answer_encode("abcdef") == "a1c1e1"
assert answer_encode("abbdcfdhe") == "a1b2c3d4e"
assert answer_shifting_letters("abc", [3, 5, 9]) == "rpl"
assert answer_shifting_letters("aaa", [1, 2, 3]) == "gfd"
for _ in range(300):
    s = random_valid(rng.randint(1, 30))
    assert answer_encode(answer_replace_digits(s)) == s
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def replaceDigits(self, s: str) -> str:
        chars = list(s)                              # chuỗi bất biến -> dùng list
        for i in range(1, len(chars), 2):           # chỉ số lẻ
            chars[i] = chr(ord(chars[i - 1]) + int(chars[i]))
        return "".join(chars)
```

Khung chung "biến đổi ký tự": `list(s)` → sửa → `''.join(...)`.

Checklist:

- [ ] Không gán `s[i] = ...` trên chuỗi.
- [ ] Dùng `int(ch)` cho chữ số, không dùng `ord(ch)` trực tiếp.
- [ ] Chỉ duyệt chỉ số lẻ `range(1, n, 2)`.
- [ ] Độ dài lẻ: chữ cuối giữ nguyên (đặc biệt khi dùng `zip`).
- [ ] Trả về **chuỗi**, không phải list.